# L31 — Ranking and Selection

**Module**: M09 | **Chapter**: 12 | **Lecture**: L31

## Learning Objectives
By the end of this notebook you will be able to:
1. Apply the Rinott procedure to select the best system with probability guarantee P*.
2. Implement indifference zone selection for k=2 and k>2 alternatives.
3. Understand why pairwise t-tests inflate family-wise Type I error.
4. Use the Nelson-Matejcic sequential screening procedure to eliminate inferior systems early.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

Problem: we have 5 nurse staffing levels (1, 2, 3, 4, 5 nurses) and want to find
the staffing level that minimises W while keeping utilisation > 60%.
We must select the best with P* = 0.90 probability.
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simpy
from scipy import stats

In [ ]:
def run_clinic_rep(n_nurses, lam, mu_reg, mu_nurse, n_patients, warmup, seed):
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    clerks = simpy.Resource(env, capacity=1)
    nurses = simpy.Resource(env, capacity=n_nurses)
    sojourns, nurse_busy = [], []

    def patient():
        t0 = env.now
        with clerks.request() as req:
            yield req
            yield env.timeout(rng.exponential(1.0 / mu_reg))
        t_nurse = env.now
        with nurses.request() as req:
            yield req
            svc = rng.exponential(1.0 / mu_nurse)
            nurse_busy.append(svc)
            yield env.timeout(svc)
        sojourns.append(env.now - t0)

    def arrivals():
        for _ in range(n_patients):
            env.process(patient())
            yield env.timeout(rng.exponential(1.0 / lam))

    env.process(arrivals())
    env.run()
    T = env.now
    rho = sum(nurse_busy) / (n_nurses * T) if T > 0 else 0
    return np.mean(sojourns[warmup:]), rho


LAM     = 5 / 60
MU_REG  = 20 / 60
MU_NRS  = 7.5 / 60
N_CUST  = 1000
WARMUP  = 100

# The k=5 alternatives
alternatives = [1, 2, 3, 4, 5]   # number of nurses

# Pilot: n0=10 replications per alternative
N0 = 10
pilot = {}
for n_n in alternatives:
    reps = [run_clinic_rep(n_n, LAM, MU_REG, MU_NRS, N_CUST, WARMUP, seed=n_n*1000+r)
            for r in range(N0)]
    ws = [r[0] for r in reps]
    rhos = [r[1] for r in reps]
    pilot[n_n] = {'W_reps': ws, 'rho_mean': np.mean(rhos)}
    print(f"n_nurses={n_n}: W̄={np.mean(ws):.2f}  s={np.std(ws,ddof=1):.3f}  ρ={np.mean(rhos):.3f}")

## 1. Naive Pairwise t-Tests: The Multiple Comparison Problem

In [ ]:
k = len(alternatives)
n_pairs = k * (k - 1) // 2
alpha_individual = 0.05
fwer = 1 - (1 - alpha_individual) ** n_pairs

print(f"k={k} alternatives → {n_pairs} pairwise comparisons")
print(f"Individual α = {alpha_individual}")
print(f"Family-wise error rate (Bonferroni upper bound) ≤ {k*(k-1)/2 * alpha_individual:.3f}")
print(f"True FWER (independent tests) ≈ {fwer:.3f}")
print()

# Bonferroni correction
alpha_bonf = alpha_individual / n_pairs
print(f"Bonferroni-corrected individual α = {alpha_bonf:.4f}")

# Run pairwise t-tests
print("\nPairwise t-tests (Bonferroni corrected):")
for i, n1 in enumerate(alternatives):
    for j, n2 in enumerate(alternatives):
        if j <= i: continue
        w1 = pilot[n1]['W_reps']
        w2 = pilot[n2]['W_reps']
        t_stat, p_val = stats.ttest_ind(w1, w2)
        sig = '*' if p_val < alpha_bonf else ' '
        print(f"  nurses={n1} vs {n2}: t={t_stat:.2f}, p={p_val:.3f} {sig}")

## 2. Rinott Procedure for Selecting the Best

The Rinott procedure guarantees:
$$P(\text{select the best}) \geq P^* = 0.90$$
when the true difference between best and second-best is at least $\delta^* = 0.5$ min.

Step 1 (pilot): compute sample variance $S_i^2$ for each system.
Step 2: compute additional replications needed: $N_i = \max\left(n_0, \left\lceil \frac{h^2 S_i^2}{\delta^{*2}} \right\rceil\right)$
where $h = h(k, \alpha, n_0)$ is the Rinott constant.

In [ ]:
P_STAR  = 0.90
DELTA_STAR = 0.5   # min — indifference zone width

# Rinott constant h (tabulated; here we use an approximation via Bonferroni)
# A common approximation: h ≈ t_{n0-1, alpha/(k-1)} for screening
# Exact table values exist in Goldsman (1983); we use a conservative approximation.
alpha_r = 1 - P_STAR   # 0.10
# Approximate Rinott constant (conservative, from Nelson 2001 tables)
h_approx = stats.t.ppf(1 - alpha_r / (2 * (k - 1)), df=N0 - 1)

print(f"P* = {P_STAR}, δ* = {DELTA_STAR} min, k = {k}")
print(f"Approximate Rinott constant h ≈ {h_approx:.3f}")
print()

Ni_required = {}
for n_n in alternatives:
    s2 = np.var(pilot[n_n]['W_reps'], ddof=1)
    Ni = max(N0, int(np.ceil(h_approx**2 * s2 / DELTA_STAR**2)))
    Ni_required[n_n] = Ni
    print(f"n_nurses={n_n}: S²={s2:.4f} → N_i = {Ni} (additional {max(0,Ni-N0)})")

print(f"\nTotal additional replications needed: {sum(max(0,N-N0) for N in Ni_required.values())}")

In [ ]:
# Run second stage
final_means = {}
for n_n in alternatives:
    Ni = Ni_required[n_n]
    all_reps = list(pilot[n_n]['W_reps'])  # already have N0
    for r in range(N0, Ni):
        w, _ = run_clinic_rep(n_n, LAM, MU_REG, MU_NRS, N_CUST, WARMUP, seed=n_n*10000+r)
        all_reps.append(w)
    final_means[n_n] = np.mean(all_reps)
    print(f"n_nurses={n_n}: N_i={Ni}, W̄={final_means[n_n]:.3f}")

best = min(final_means, key=final_means.get)
print(f"\nSelected best: n_nurses = {best}  (W̄ = {final_means[best]:.3f} min)")
print(f"Nurse utilisation: {pilot[best]['rho_mean']:.3f}  {'✓ > 0.60' if pilot[best]['rho_mean'] > 0.60 else '✗ < 0.60'}")

## 3. Visualise the Selection Decision

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Mean W by alternative
xs = list(final_means.keys())
ys = [final_means[n] for n in xs]
rhos = [pilot[n]['rho_mean'] for n in xs]

colors = ['gold' if n == best else 'steelblue' for n in xs]
ax1.bar(xs, ys, color=colors, alpha=0.8, edgecolor='black')
ax1.set_xlabel('Number of nurses')
ax1.set_ylabel('Mean sojourn W (min)')
ax1.set_title(f'Rinott selection: best = {best} nurse(s)')
ax1.grid(True, axis='y', alpha=0.3)

# Utilisation by alternative
ax2.bar(xs, rhos, color=colors, alpha=0.8, edgecolor='black')
ax2.axhline(0.60, color='red', lw=1.5, linestyle='--', label='ρ=0.60 threshold')
ax2.set_xlabel('Number of nurses')
ax2.set_ylabel('Nurse utilisation ρ')
ax2.set_title('Nurse utilisation by staffing level')
ax2.legend()
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Sequential Screening: Eliminate Dominated Systems Early

Screening saves simulation budget by removing clearly inferior systems
early in the pilot, before running expensive second-stage replications.

In [ ]:
def screen_alternatives(pilot_reps, delta_star, alpha):
    """
    Kim-Nelson (2001) screening: eliminate system i if
    W_bar_i - W_bar_best > delta_star + critical_value.
    Returns the set of surviving (non-dominated) systems.
    """
    n0   = len(next(iter(pilot_reps.values())))
    keys = list(pilot_reps.keys())
    bars = {k: np.mean(v) for k, v in pilot_reps.items()}

    # Pairwise KN bound
    surviving = set(keys)
    for i in keys:
        for j in keys:
            if i == j or i not in surviving: continue
            diff = bars[i] - bars[j]
            s2_ij = np.var(np.array(pilot_reps[i]) - np.array(pilot_reps[j]), ddof=1)
            eta = 0.5 * (stats.t.ppf(1 - alpha / (k-1), df=n0-1))**2 * s2_ij / delta_star**2
            if diff > eta:
                surviving.discard(i)
    return surviving


pilot_reps = {n: pilot[n]['W_reps'] for n in alternatives}
surviving = screen_alternatives(pilot_reps, delta_star=DELTA_STAR, alpha=0.10)

print(f"After screening: surviving alternatives = {sorted(surviving)}")
print(f"Eliminated: {set(alternatives) - surviving}")
print(f"\nBudget saving: {len(set(alternatives)-surviving)} systems eliminated early.")
print(f"Only {len(surviving)} systems need the expensive second stage.")

---
## Try It Yourself

1. **P* sensitivity**: Repeat the Rinott procedure with P*=0.80 and P*=0.95. How does the required total replication count change? Plot total replications vs P* for P* ∈ {0.80, 0.85, 0.90, 0.95, 0.99}.

2. **δ* choice**: The indifference zone δ*=0.5 min means we don't care if one system is within 0.5 min of the best. What happens if the decision maker says "any difference > 0.1 min matters"? Compute the new Ni for each system and the total budget increase.

3. **Feasibility constraint**: We required ρ > 0.60 (nurse utilisation). Suppose the constraint is tightened to ρ > 0.70. Does this change which alternative is selected? What happens to the nurse utilisation at the selected level — does the trade-off between W and ρ create a Pareto frontier?